# Nemotron Reasoning Challenge: getting to know the data

I'm working on this competition, and before I went anywhere near modeling I wanted to just sit with the data for a bit. So this is me poking around `train.csv`: how many puzzles there are, what the different puzzle types actually look like, how the answers are formatted, and how a typical prompt is put together.

Nothing fancy here. I'm just getting my bearings, and figured I'd share it in case it helps you do the same.

In [ ]:
import os, glob, re, textwrap
import pandas as pd
import matplotlib.pyplot as plt

# Kaggle mounts competition files under /kaggle/input; find train.csv wherever it lands.
path = next(p for p in glob.glob("/kaggle/input/**/train.csv", recursive=True))
df = pd.read_csv(path)
print("rows:", len(df))
df.head(3)

## What kinds of puzzles are in here?

The prompts aren't all the same. They come from a few different puzzle templates, and each one opens with a sentence you can recognize. So I bucketed every row by that opening line to see how things break down.

In [ ]:
KEY = {
    "bit_manipulation": "secret bit manipulation rule",
    "cipher": "secret encryption rules are used on text",
    "numeral": "numbers are secretly converted into a different numeral system",
    "unit_conversion": "secret unit conversion is applied to measurements",
    "gravity": "gravitational constant has been secretly changed",
    "transformation": "secret set of transformation rules is applied to equations",
}
def family(p):
    for k, kw in KEY.items():
        if kw in p:
            return k
    return "other"
df["family"] = df["prompt"].map(family)
counts = df["family"].value_counts()
print(counts)

In [ ]:
ax = counts.sort_values().plot.barh(figsize=(7,3.5))
ax.set_title("Puzzles per type (train)")
ax.set_xlabel("count")
plt.tight_layout(); plt.show()

## What does each one actually look like?

Here's one prompt from each type, so you can see the shape of them. They all give you a few worked examples first, then ask you to fill in the answer for a new one.

In [ ]:
for fam in counts.index:
    row = df[df["family"] == fam].iloc[0]
    print("="*70)
    print(fam, "| answer:", repr(row["answer"]))
    print("-"*70)
    print(textwrap.fill(row["prompt"], 100)[:600])
    print()

## How are the answers formatted?

This is the part that surprised me a little. The answers are all over the place depending on the puzzle. Some are binary strings, some are plain numbers, some are Roman numerals or words, and a few are short symbolic strings. I wanted to see how that splits up, and how long the answers tend to run.

In [ ]:
def kind(a):
    a = str(a)
    if re.fullmatch(r"[01]+", a): return "binary"
    try:
        float(a); return "numeric"
    except: pass
    if re.fullmatch(r"[A-Za-z]+", a): return "alpha"
    return "other/symbolic"
df["answer_kind"] = df["answer"].map(kind)
pd.crosstab(df["family"], df["answer_kind"])

In [ ]:
df["ans_len"] = df["answer"].astype(str).str.len()
df.groupby("family")["ans_len"].describe()[["mean","min","max"]].round(2)

## How long are the prompts?

I also looked at how long the prompts run, in characters. It's a rough way to guess how many worked examples each puzzle hands you before the real question shows up.

In [ ]:
df["prompt_len"] = df["prompt"].str.len()
ax = df.boxplot(column="prompt_len", by="family", rot=45, figsize=(8,4))
plt.title("Prompt length by type"); plt.suptitle(""); plt.ylabel("characters")
plt.tight_layout(); plt.show()

## What I took away

A few things stood out to me. The data is split pretty evenly across the six puzzle types, so no single one dominates. The answer formats really do change from one type to the next, and I think that's going to matter a lot for anything graded on exact matches. And every prompt is few-shot, so you get a handful of examples and then a query to solve.

If you're picking this up, the next thing I'd look at is accuracy broken down by puzzle type rather than one number for everything. The types are different enough that an overall score can quietly hide where the real room to improve actually is.

## Where I'm taking this

This notebook is just the warm-up. Off to the side I've taken it quite a bit further than what's here.

I've set up my own evaluation so I can score a model per puzzle type instead of staring at one lumped number, and I've been running fine-tuning rounds and iterating off what those breakdowns tell me. It's basically a measure, tune, measure again loop, and the per-type results are what I use to decide where to put my effort next.

I'm going to keep the actual approach and the numbers to myself for now, since it's a live competition and that's kind of the whole game. But I wanted to put down a marker that there's real strategy work happening behind this, not just the EDA you see above. If you're also in this one, I'm happy to trade notes at a high level.